## How many patients?

Cerner won't list every patient or give a total, so this counts patients by last name, following the "next page" links. Add more names to `NAMES` to widen the count.

In [5]:
NAMES = ["SMART", "PETERS", "WILMA", "TIMMY"]

all_ids = set()
for name in NAMES:
    url, params = f"{OPEN_URL}/Patient", {"name": name, "_count": 100}
    while url:
        page = requests.get(url, params=params, headers=HEADERS).json()
        for entry in page.get("entry", []):
            all_ids.add(entry["resource"]["id"])
        url = next((l["url"] for l in page.get("link", []) if l["relation"] == "next"), None)
        params = None  # the next URL already has the params
    print(name, "-> running total:", len(all_ids))

print("Unique patients found:", len(all_ids))

NameError: name 'OPEN_URL' is not defined

## Save a few patients as pandas DataFrames

One table per record type. Each row is one record; nested JSON is flattened into columns like `code.text` or `valueQuantity.value`. Tables are also saved as CSVs in `data/` so you can open them in Excel.

In [ ]:
import pandas as pd

patient_ids = [e["resource"]["id"] for e in bundle["entry"][:3]]  # first 3 patients from the search
RESOURCES = ["Patient", "Encounter", "Condition", "Observation", "MedicationRequest", "AllergyIntolerance"]

dfs = {}
for resource in RESOURCES:
    rows = []
    for pid in patient_ids:
        params = {"_id": pid} if resource == "Patient" else {"patient": pid, "_count": 50}
        r = requests.get(f"{OPEN_URL}/{resource}", params=params, headers=HEADERS)
        rows += [e["resource"] for e in r.json().get("entry", [])]  # .get: skip if the sandbox times out
    dfs[resource] = pd.json_normalize(rows)
    print(f"{resource}: {len(dfs[resource])} rows, {len(dfs[resource].columns)} columns")

os.makedirs("data", exist_ok=True)
for name, df in dfs.items():
    df.to_csv(f"data/{name}.csv", index=False)

Patient: 3 rows, 20 columns
Encounter: 25 rows, 21 columns
Condition: 11 rows, 19 columns
Observation: 65 rows, 28 columns
MedicationRequest: 50 rows, 32 columns
AllergyIntolerance: 88 rows, 28 columns


In [ ]:
# Look at one table. Change the name to explore the others.
df = dfs["Observation"]
print(list(df.columns))
df.head()

['resourceType', 'id', 'identifier', 'status', 'category', 'effectiveDateTime', 'issued', 'performer', 'component', 'meta.versionId', 'meta.lastUpdated', 'text.status', 'text.div', 'code.coding', 'code.text', 'subject.reference', 'encounter.reference', 'valueQuantity.extension', 'valueQuantity.value', 'valueQuantity.unit', 'valueQuantity.system', 'valueQuantity.code', 'hasMember', 'valueCodeableConcept.coding', 'valueCodeableConcept.text', 'valueString', 'note', 'effectivePeriod.start']


,resourceType,id,identifier,status,category,effectiveDateTime,issued,performer,component,meta.versionId,...,valueQuantity.value,valueQuantity.unit,valueQuantity.system,valueQuantity.code,hasMember,valueCodeableConcept.coding,valueCodeableConcept.text,valueString,note,effectivePeriod.start
0,Observation,BP-209135312-209135314,"[{'system': 'https://fhir.cerner.com/ceuuid', ...",final,[{'coding': [{'system': 'http://terminology.hl...,2026-09-29T07:10:34.000Z,2026-09-29T07:10:34.000Z,[{'extension': [{'valueCodeableConcept': {'cod...,[{'code': {'coding': [{'system': 'https://fhir...,1-1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Observation,VS-209135310,"[{'system': 'https://fhir.cerner.com/ceuuid', ...",final,[{'coding': [{'system': 'http://terminology.hl...,2026-09-29T07:10:34.000Z,2026-09-29T07:10:34.000Z,[{'extension': [{'valueCodeableConcept': {'cod...,NaN,1,...,37.6,degC,http://unitsofmeasure.org,Cel,NaN,NaN,NaN,NaN,NaN,NaN
2,Observation,VS-209135308,"[{'system': 'https://fhir.cerner.com/ceuuid', ...",final,[{'coding': [{'system': 'http://terminology.hl...,2026-09-29T07:10:34.000Z,2026-09-29T07:10:34.000Z,[{'extension': [{'valueCodeableConcept': {'cod...,NaN,1,...,15.0,br/min,http://unitsofmeasure.org,/min,NaN,NaN,NaN,NaN,NaN,NaN
3,Observation,VS-209135306,"[{'system': 'https://fhir.cerner.com/ceuuid', ...",final,[{'coding': [{'system': 'http://terminology.hl...,2026-09-29T07:10:34.000Z,2026-09-29T07:10:34.000Z,[{'extension': [{'valueCodeableConcept': {'cod...,NaN,1,...,96.0,%,http://unitsofmeasure.org,%,NaN,NaN,NaN,NaN,NaN,NaN
4,Observation,VS-209135304,"[{'system': 'https://fhir.cerner.com/ceuuid', ...",final,[{'coding': [{'system': 'http://terminology.hl...,2026-09-29T07:10:34.000Z,2026-09-29T07:10:34.000Z,[{'extension': [{'valueCodeableConcept': {'cod...,NaN,1,...,78.0,bpm,http://unitsofmeasure.org,/min,NaN,NaN,NaN,NaN,NaN,NaN


# Explore Cerner sandbox FHIR data

Run cells top to bottom. Part 1 needs no login. Part 2 uses your app secret.

In [ ]:
import os, json, requests

TENANT = "ec2458f2-1e24-41c8-b71b-0e701af7583d"  # Cerner public sandbox tenant
HEADERS = {"Accept": "application/fhir+json"}

def show(data):
    print(json.dumps(data, indent=2)[:3000])  # first 3000 chars is enough to see the shape

## Part 1 - Open sandbox (no auth, fake patients)

In [ ]:
OPEN_URL = f"https://fhir-open.cerner.com/r4/{TENANT}"

r = requests.get(f"{OPEN_URL}/Patient", params={"family": "SMART", "_count": 5}, headers=HEADERS)
bundle = r.json()
show(bundle)

{
  "resourceType": "Bundle",
  "id": "993d96ec-971a-4818-ae9f-1e8cd5b3d629",
  "type": "searchset",
  "link": [
    {
      "relation": "self",
      "url": "https://fhir-open.cerner.com/r4/ec2458f2-1e24-41c8-b71b-0e701af7583d/Patient?family=SMART&_count=5"
    },
    {
      "relation": "next",
      "url": "https://fhir-open.cerner.com/r4/ec2458f2-1e24-41c8-b71b-0e701af7583d/Patient?-pageContext=1328a01c-b60f-4ece-aeae-1d08186f5fc3&-pageDirection=NEXT"
    }
  ],
  "entry": [
    {
      "fullUrl": "https://fhir-open.cerner.com/r4/ec2458f2-1e24-41c8-b71b-0e701af7583d/Patient/12742399",
      "resource": {
        "resourceType": "Patient",
        "id": "12742399",
        "meta": {
          "versionId": "210414",
          "lastUpdated": "2026-07-28T14:51:15.000Z"
        },
        "text": {
          "status": "generated",
          "div": "<div xmlns=\"http://www.w3.org/1999/xhtml\"><p><b>Patient</b></p><p><b>Name</b>: SMART II, SANDY</p><p><b>Status</b>: Active</p><p><b>DOB</b

In [ ]:
# Pick the first patient and look at the data a transfer summary would use
patient_id = bundle["entry"][0]["resource"]["id"]
print("Patient:", patient_id)

for resource in ["Condition", "Encounter", "Observation", "MedicationRequest", "AllergyIntolerance"]:
    r = requests.get(f"{OPEN_URL}/{resource}", params={"patient": patient_id, "_count": 3}, headers=HEADERS)
    print(f"\n===== {resource} ({r.status_code}) =====")
    show(r.json())

Patient: 12742399

===== Condition (504) =====
{
  "message": "Gateway Timeout",
  "code": 504
}

===== Encounter (200) =====
{
  "resourceType": "Bundle",
  "id": "6146efd4-be00-470c-92b5-373521bb0ac3",
  "type": "searchset",
  "link": [
    {
      "relation": "self",
      "url": "https://fhir-open.cerner.com/r4/ec2458f2-1e24-41c8-b71b-0e701af7583d/Encounter?patient=12742399&_count=3"
    },
    {
      "relation": "next",
      "url": "https://fhir-open.cerner.com/r4/ec2458f2-1e24-41c8-b71b-0e701af7583d/Encounter?patient=12742399&-pageContext=bb60723b-b2e2-4092-8769-79d5fa7f27b8&-pageDirection=NEXT"
    }
  ],
  "entry": [
    {
      "fullUrl": "https://fhir-open.cerner.com/r4/ec2458f2-1e24-41c8-b71b-0e701af7583d/Encounter/97955594",
      "resource": {
        "resourceType": "Encounter",
        "id": "97955594",
        "meta": {
          "versionId": "0",
          "lastUpdated": "2020-11-05T18:49:47.000Z"
        },
        "text": {
          "status": "generated",
        

In [ ]:
# Bed info lives in Location resources. Cerner requires a filter: -physicalType=bd means "bed"
r = requests.get(f"{OPEN_URL}/Location", params={"-physicalType": "bd", "_count": 5}, headers=HEADERS)
show(r.json())

{
  "resourceType": "Bundle",
  "id": "78d9a4cd-6987-4a0f-ab1f-66865845abd6",
  "type": "searchset",
  "link": [
    {
      "relation": "self",
      "url": "https://fhir-open.cerner.com/r4/ec2458f2-1e24-41c8-b71b-0e701af7583d/Location?-physicalType=bd&_count=5"
    },
    {
      "relation": "next",
      "url": "https://fhir-open.cerner.com/r4/ec2458f2-1e24-41c8-b71b-0e701af7583d/Location?-physicalType=bd&_count=5&-offset=5"
    }
  ],
  "entry": [
    {
      "fullUrl": "https://fhir-open.cerner.com/r4/ec2458f2-1e24-41c8-b71b-0e701af7583d/Location/2575264761",
      "resource": {
        "resourceType": "Location",
        "id": "2575264761",
        "meta": {
          "versionId": "0",
          "lastUpdated": "2023-12-05T11:31:40Z"
        },
        "text": {
          "status": "generated",
          "div": "<div xmlns=\"http://www.w3.org/1999/xhtml\"><p><b>Location</b></p><p><b>Name</b>: 1</p><p><b>Alias</b>: 1</p><p><b>Managing Organization</b>: Model Clinic 2</p><p><b>Part 

## Readable version

Every FHIR record has a `text.div` field: a short human-readable summary written by Cerner. Stripping the HTML gives one line per record, which is a good input for the AI summary.

In [ ]:
import re

def readable(resource):
    html = resource.get("text", {}).get("div", "")
    return re.sub(r"<[^>]+>", " ", html).split()  # strip HTML tags

for resource in ["Patient", "Encounter", "Condition", "Observation", "MedicationRequest", "AllergyIntolerance"]:
    params = {"_id": patient_id} if resource == "Patient" else {"patient": patient_id, "_count": 10}
    r = requests.get(f"{OPEN_URL}/{resource}", params=params, headers=HEADERS)
    print(f"\n===== {resource} ({r.status_code}) =====")
    for entry in r.json().get("entry", []):
        print("-", " ".join(readable(entry["resource"])))


===== Patient (200) =====
- Patient Name : SMART II, SANDY Status : Active DOB : Sep 15, 2020 Administrative Gender : Unknown Marital Status : Married

===== Encounter (200) =====
- Encounter Patient : SMART II, SANDY Location : Model Urgent Care Type : Outpatient Class : ambulatory Status : In Progress Service Provider : Model Urgent Care
- Encounter Patient : SMART II, SANDY Location : Model Urgent Care Type : Outpatient Class : ambulatory Status : In Progress Service Provider : Model Urgent Care
- Encounter Patient : SMART II, SANDY Location : Model Urgent Care Type : Outpatient Class : ambulatory Status : In Progress Service Provider : Model Urgent Care
- Encounter Patient : SMART II, SANDY Location : Model Urgent Care Type : Outpatient Class : ambulatory Status : In Progress Service Provider : Model Urgent Care
- Encounter Patient : SMART II, SANDY Location : Model Urgent Care Type : Outpatient Class : ambulatory Status : In Progress Service Provider : Model Urgent Care
- Encount

## Part 2 - Your app (DischargeAgent) with the secret

The next cell pops up a box asking for the secret. Paste it there; it is not saved in the notebook.

In [ ]:
CLIENT_ID = "92e1d486-4f2d-4218-81b8-cbeb9d3748f8"  # DischargeAgent_2 (System app) Client ID
from getpass import getpass
SECRET = 'hLGNk5AMvHrD9a9IFOlj5ump59QnbaXx'
TOKEN_URL = f"https://authorization.cerner.com/tenants/ec2458f2-1e24-41c8-b71b-0e701af7583d/protocols/oauth2/profiles/smart-v1/token"
SECURE_URL = f"https://fhir-ehr-code.cerner.com/r4/ec2458f2-1e24-41c8-b71b-0e701af7583d"

r = requests.post(
    TOKEN_URL,
    auth=(CLIENT_ID, SECRET),
    data={"grant_type": "client_credentials", "scope": "system/Patient.read system/Encounter.read system/Condition.read"},
)
print(r.status_code)
show(r.json())
token = r.json().get("access_token")

NameError: name 'requests' is not defined

In [ ]:
auth_headers = {**HEADERS, "Authorization": f"Bearer {token}"}
r = requests.get(f"{SECURE_URL}/Patient/{patient_id}", headers=auth_headers)
print(r.status_code)
show(r.json())

NameError: name 'token' is not defined